# 🎭 HY-Motion 1.0 — Text to 3D Human Motion

Generate realistic 3D human motion animations from text prompts using Tencent's HunyuanMotion model.

## Quick Start
1. Open this notebook in **VS Code** with the **Google Colab extension** installed
2. Click **Select Kernel** → **Colab** → Choose a **GPU runtime** (T4 or A100)
3. Run **Cell 1** (Setup) — installs everything (~2 min first time)
4. Run **Cell 2** (Download Models) — downloads AI models (~3 min first time)
5. Run **Cell 3** (Load Model) — loads the model onto GPU (~30s)
6. Run **Cell 4** (Generate!) — type your prompt and watch the magic ✨
7. Run **Cell 5** (View) — see the interactive 3D animation

## Requirements
- VS Code with [Google Colab extension](https://marketplace.visualstudio.com/items?itemName=google.colab)
- A Google account with access to Colab GPU runtimes
- That's it! Everything else is installed automatically.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                        CELL 1: SETUP & INSTALL                             ║
# ║  Clones the HY-Motion repo and installs all Python dependencies.           ║
# ║  Safe to re-run — it skips steps that are already done.                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os

# --- Clone the repo (only if it doesn't exist yet) ---
if not os.path.isdir('/content/HY-Motion-1.0/hymotion'):
    !git clone https://github.com/Tencent-Hunyuan/HY-Motion-1.0.git /content/HY-Motion-1.0
    print('✅ Repository cloned!')
else:
    print('✅ Repository already exists, skipping clone.')

# --- Always set the working directory (idempotent) ---
os.chdir('/content/HY-Motion-1.0')
print(f'📂 Working directory: {os.getcwd()}')

# --- Install Python dependencies ---
# These are carefully selected to avoid conflicts with Colab's pre-installed PyTorch/CUDA.
# We do NOT install torch/torchvision (Colab already has GPU-optimized versions).
# We do NOT install fbxsdkpy (requires C++ compilation that fails on Colab).
# We do NOT install PyYAML==6.0 (conflicts with Colab's Python 3.12).
!pip install -q \
    huggingface_hub>=0.33.5 \
    torchdiffeq==0.2.5 \
    accelerate==0.30.1 \
    diffusers==0.26.3 \
    transformers==4.53.3 \
    einops==0.8.1 \
    safetensors==0.5.3 \
    bitsandbytes==0.49.0 \
    transforms3d==0.4.2 \
    omegaconf \
    click>=8.2.1

# --- Verify GPU ---
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'🚀 GPU detected: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('⚠️  No GPU detected! Go to Colab settings and select a GPU runtime.')
    print('   Without a GPU, generation will be extremely slow.')

print('\n✅ Setup complete!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                      CELL 2: DOWNLOAD AI MODELS                            ║
# ║  Downloads 3 models from HuggingFace. Takes ~3 min on first run.           ║
# ║  Subsequent runs are instant (cached).                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os
os.chdir('/content/HY-Motion-1.0')  # Ensure correct directory

# 1. HY-Motion-1.0-Lite — The main motion generation model (~1.8 GB)
if not os.path.isfile('ckpts/tencent/HY-Motion-1.0-Lite/latest.ckpt'):
    print('📥 Downloading HY-Motion-1.0-Lite (~1.8 GB)...')
    !huggingface-cli download tencent/HY-Motion-1.0 --include "HY-Motion-1.0-Lite/*" --local-dir ckpts/tencent
else:
    print('✅ HY-Motion-1.0-Lite already downloaded.')

# 2. CLIP ViT-L/14 — Text understanding model (~1.7 GB)
if not os.path.isfile('ckpts/clip-vit-large-patch14/config.json'):
    print('📥 Downloading CLIP ViT-L/14 (~1.7 GB)...')
    !huggingface-cli download openai/clip-vit-large-patch14 --local-dir ckpts/clip-vit-large-patch14
else:
    print('✅ CLIP ViT-L/14 already downloaded.')

# 3. Qwen3-8B (4-bit quantized) — Large language model for text encoding (~5 GB)
if not os.path.isfile('ckpts/Qwen3-8B/config.json'):
    print('📥 Downloading Qwen3-8B-bnb-4bit (~5 GB)...')
    !huggingface-cli download unsloth/Qwen3-8B-bnb-4bit --local-dir ckpts/Qwen3-8B-bnb-4bit
    # Rename to match the expected path in the code
    if os.path.isdir('ckpts/Qwen3-8B-bnb-4bit') and not os.path.isdir('ckpts/Qwen3-8B'):
        !mv ckpts/Qwen3-8B-bnb-4bit ckpts/Qwen3-8B
else:
    print('✅ Qwen3-8B already downloaded.')

print('\n✅ All models ready!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                    CELL 3: LOAD MODEL ONTO GPU                             ║
# ║  Loads the motion generation pipeline and applies speed optimizations.     ║
# ║  Takes ~30 seconds. Only needs to run once per session.                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, time, torch
os.chdir('/content/HY-Motion-1.0')  # Ensure correct directory
os.environ['USE_HF_MODELS'] = '0'  # Use local model files, not HuggingFace Hub

from hymotion.utils.t2m_runtime import T2MRuntime

print('🔄 Loading model onto GPU (this takes ~30 seconds)...')
t0 = time.time()
runtime = T2MRuntime(
    config_path='ckpts/tencent/HY-Motion-1.0-Lite/config.yml',
    ckpt_name='ckpts/tencent/HY-Motion-1.0-Lite/latest.ckpt',
    skip_text=False,
    disable_prompt_engineering=True,  # Skip the huge PromptRewriter model (saves ~10GB RAM)
)
load_time = time.time() - t0

# ┌──────────────────────────────────────────────────────────────────────────────┐
# │                    ⚡ SPEED OPTIMIZATION SETTINGS ⚡                        │
# │                                                                              │
# │  The single most impactful optimization: reduce ODE sampling steps.          │
# │  The default is 50 steps. We reduce to 10 for a ~3x speedup.                │
# │                                                                              │
# │  ╔════════════════════════════════════════════════════════════╗               │
# │  ║  IF YOUR MOTION QUALITY LOOKS BAD, INCREASE THIS NUMBER  ║               │
# │  ╚════════════════════════════════════════════════════════════╝               │
# │                                                                              │
# │  Benchmarks on T4 GPU (3-second duration):                                  │
# │    50 steps  →  ~10s  (highest quality, slowest)                             │
# │    25 steps  →   ~6s  (great quality)                                        │
# │    15 steps  →  ~4.4s (good quality)                                         │
# │    10 steps  →  ~3.6s (good quality, recommended)                            │
# │     5 steps  →  ~2.9s (fastest, may lose some detail)                        │
# └──────────────────────────────────────────────────────────────────────────────┘

pipeline = runtime.pipelines[0]

# ████████████████████████████████████████████████████████████████
# ██                                                          ██
# ██   ODE_STEPS: Change this to control speed vs quality     ██
# ██                                                          ██
ODE_STEPS = 10  # ◀◀◀ CHANGE ME! (range: 5-50, default: 50)  ██
# ██                                                          ██
# ████████████████████████████████████████████████████████████████

pipeline.validation_steps = ODE_STEPS

print(f'\n✅ Model loaded in {load_time:.1f}s')
print(f'🚀 GPU: {torch.cuda.get_device_name(0)}')
print(f'⚡ ODE steps: {ODE_STEPS} (default=50, lower=faster)')
print(f'\n🎭 Ready to generate! Run Cell 4 below.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                      CELL 4: GENERATE 3D MOTION                            ║
# ║  Type your prompt below and run this cell to generate a 3D animation!      ║
# ║  Re-run this cell as many times as you want with different prompts.         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, time, torch

# ████████████████████████████████████████████████████████████████████████████████
# ██                                                                          ██
# ██   PROMPT: Describe the motion you want to generate                       ██
# ██                                                                          ██
PROMPT = "A person dancing like a ballerina"  # ◀◀◀ CHANGE ME!               ██
# ██                                                                          ██
# ████████████████████████████████████████████████████████████████████████████████

# ┌──────────────────────────────────────────────────────────────────────────────┐
# │                     ⚙️  GENERATION SETTINGS ⚙️                              │
# │                                                                              │
# │  DURATION: Length of the animation in seconds (max ~12s)                     │
# │  SEED:     Random seed for reproducibility (same seed = same motion)         │
# │  CFG:      Classifier-free guidance scale (higher = more prompt-faithful)    │
# │                                                                              │
# │  ╔════════════════════════════════════════════════════════════╗               │
# │  ║  IF THE MOTION DOESN'T MATCH YOUR PROMPT WELL ENOUGH,   ║               │
# │  ║  TRY INCREASING CFG_SCALE TO 7.0 OR 10.0                ║               │
# │  ╚════════════════════════════════════════════════════════════╝               │
# └──────────────────────────────────────────────────────────────────────────────┘

DURATION = 3.0    # ◀◀◀ Animation length in seconds (0.5 to 12.0)
SEED = 42         # ◀◀◀ Random seed (change for different variations)
CFG_SCALE = 5.0   # ◀◀◀ Guidance scale (1.0=creative, 5.0=balanced, 10.0=strict)

# ──────────────────────── Generation ────────────────────────────
print(f'🎭 Generating: "{PROMPT}"')
print(f'   Duration: {DURATION}s | Seed: {SEED} | CFG: {CFG_SCALE} | Steps: {pipeline.validation_steps}')
print(f'   Generating...')

torch.cuda.synchronize()
t0 = time.time()

# Generate the motion (output_dir=None lets the internal code handle paths correctly)
html_content, _, _ = runtime.generate_motion(
    text=PROMPT,
    seeds_csv=str(SEED),
    duration=DURATION,
    cfg_scale=CFG_SCALE,
    output_format='dict',
    output_dir=None,
)

torch.cuda.synchronize()
gen_time = time.time() - t0

# ──────────────────────── Save to organized folder ──────────────
# Each prompt gets its own subfolder: output/<prompt>/<prompt>.html
target_dir = os.path.join('output', PROMPT)
os.makedirs(target_dir, exist_ok=True)
html_path = os.path.join(target_dir, f'{PROMPT}.html')

with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f'\n✅ Done in {gen_time:.1f}s!')
print(f'💾 Saved to: {html_path}')
print(f'\n▶️  Run Cell 5 below to see the 3D animation!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                    CELL 5: VIEW 3D ANIMATION                               ║
# ║  Displays the generated motion as an interactive 3D viewer.                ║
# ║  You can click and drag to rotate the camera!                              ║
# ║                                                                            ║
# ║  The .html file saved in Cell 4 is a standalone 3D video.                  ║
# ║  You can download it and open it in any browser, even offline!             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import base64
import IPython.display as display

# Encode the HTML content and display it safely inside an iframe
b64_html = base64.b64encode(html_content.encode('utf-8')).decode('utf-8')

display.display(display.HTML(f'''
<div style="text-align: center; margin: 10px 0;">
    <h3 style="font-family: sans-serif; color: #333;">🎭 {PROMPT}</h3>
    <p style="font-family: sans-serif; color: #666; font-size: 13px;">
        Generated in {gen_time:.1f}s | {DURATION}s duration | Seed {SEED} | CFG {CFG_SCALE} | {pipeline.validation_steps} steps
    </p>
</div>
<iframe 
    src="data:text/html;base64,{b64_html}" 
    width="100%" 
    height="550px"
    style="border: 2px solid #ddd; border-radius: 12px; background: white; box-shadow: 0 4px 12px rgba(0,0,0,0.1);">
</iframe>
<p style="text-align: center; font-family: sans-serif; color: #999; font-size: 12px; margin-top: 8px;">
    💡 Click and drag to rotate the camera | Scroll to zoom | The saved .html file works offline in any browser
</p>
'''))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                  CELL 6: INSTALL SOMA RETARGETER                             ║
# ║  Installs the NVIDIA/soma-retargeter library to convert our motion.          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os

if not os.path.isdir('/content/soma-retargeter'):
    print("📥 Cloning NVIDIA/soma-retargeter...")
    !git clone https://github.com/NVIDIA/soma-retargeter.git /content/soma-retargeter
    os.chdir('/content/soma-retargeter')
    print("📦 Installing soma-retargeter via pip...")
    # Intentionally ignoring dependency resolution errors as Colab has preinstalled packages
    !pip install --no-dependencies -e .
    !pip install warp-lang scipy
    print("✅ Installation complete!")
else:
    print("✅ soma-retargeter already installed.")
    
os.chdir('/content/HY-Motion-1.0')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                  CELL 7: CONVERT SMPL TO SOMA BVH                            ║
# ║  Maps the 22-joint HY-Motion output (rot6d) to the SOMA humanoid skeleton    ║
# ║  and saves a compliant .bvh file for the retargeter.                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os
import torch
import numpy as np
from scipy.spatial.transform import Rotation

print("🔄 Converting HY-Motion output to SOMA BVH format...")

# 1. Provide the mapping from the 66 SOMA BVH joints to the 22 SMPL joints
# Standard SMPL 22-joint indices:
smpl_to_soma = {
    'Hips': 0, 'LeftLeg': 1, 'RightLeg': 2, 'Spine1': 3,
    'LeftShin': 4, 'RightShin': 5, 'Spine2': 6, 'LeftFoot': 7,
    'RightFoot': 8, 'Chest': 9, 'LeftToeBase': 10, 'RightToeBase': 11,
    'Neck1': 12, 'LeftShoulder': 13, 'RightShoulder': 14, 'Head': 15,
    'LeftArm': 16, 'RightArm': 17, 'LeftForeArm': 18, 'RightForeArm': 19,
    'LeftHand': 20, 'RightHand': 21
}

# 2. Extract motion from HY-Motion's output dict (`model_output` from Cell 4)
rot6d = model_output['rot6d_smooth'][0].cpu().numpy()  # [frames, 22, 6]
transl = model_output['transl_smooth'][0].cpu().numpy() * 100.0  # [frames, 3] convert to cm
frames = rot6d.shape[0]

# Convert rot6d to rotation matrices
x = rot6d[..., :3]
y = rot6d[..., 3:]
x = x / np.linalg.norm(x, axis=-1, keepdims=True)
z = np.cross(x, y)
z = z / np.linalg.norm(z, axis=-1, keepdims=True)
y = np.cross(z, x)
rot_mats = np.stack([x, y, z], axis=-1)  # [frames, 22, 3, 3]

# Convert to Euler angles 'ZYX' (extrinsic ZYX maps to BVH 'Zrotation Yrotation Xrotation')
euler_angles = Rotation.from_matrix(rot_mats).as_euler('ZYX', degrees=True) # [frames, 22, 3]

# 3. Read the exact SOMA hierarchy header from the cloned repo
template_bvh_path = '/content/soma-retargeter/soma_retargeter/configs/soma/soma_zero_frame0.bvh'
with open(template_bvh_path, 'r') as f:
    template_lines = f.readlines()

header_lines = []
joint_names = []
for line in template_lines:
    if line.startswith('MOTION'):
        break
    header_lines.append(line)
    if line.strip().startswith("ROOT") or line.strip().startswith("JOINT"):
        joint_names.append(line.strip().split()[-1])

header_text = "".join(header_lines)

# 4. Write the converted .bvh file
OUTPUT_BVH_PATH = '/content/soma-retargeter/assets/motions/bvh/generated.bvh'
os.makedirs(os.path.dirname(OUTPUT_BVH_PATH), exist_ok=True)

with open(OUTPUT_BVH_PATH, 'w') as f:
    f.write(header_text)
    f.write("MOTION\n")
    f.write(f"Frames: {frames}\n")
    f.write("Frame Time: 0.0333333\n")  # 30 fps
    
    for frame_idx in range(frames):
        frame_data = []
        for joint in joint_names:
            if joint == 'Hips':
                frame_data.extend(transl[frame_idx].tolist())
            else:
                frame_data.extend([0.0, 0.0, 0.0])
                
            if joint in smpl_to_soma:
                smpl_idx = smpl_to_soma[joint]
                frame_data.extend(euler_angles[frame_idx, smpl_idx].tolist())
            else:
                frame_data.extend([0.0, 0.0, 0.0])
        f.write(" ".join(map(str, frame_data)) + "\n")

print(f"✅ Successfully converted HY-Motion output to SOMA BVH!")
print(f"📁 Saved to: {OUTPUT_BVH_PATH}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                  CELL 8: RETARGET TO UNITREE G1                              ║
# ║  Runs NVIDIA Soma-Retargeter to translate the BVH to G1 robot joint angles   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os
import json

print("🔄 Running SOMA Retargeter for Unitree G1...")

# 1. Create headless configuration
config = {
    "import_folder": "assets/motions/bvh",
    "export_folder": "assets/motions/export_g1",
    "batch_size": 1,
    "retargeter": "Newton",
    "retarget_source": "soma",
    "retarget_target": "unitree_g1",
    "retarget_source_facing_direction": "Mujoco"
}

os.chdir('/content/soma-retargeter')
with open('colab_config.json', 'w') as f:
    json.dump(config, f, indent=4)

# 2. Run the headless converter
!python app/bvh_to_csv_converter.py --config colab_config.json --viewer null

OUTPUT_CSV_PATH = '/content/soma-retargeter/assets/motions/export_g1/generated.csv'
if os.path.exists(OUTPUT_CSV_PATH):
    print(f"\n✅ Successfully retargeted to Unitree G1!")
    print(f"🦾 Output motion CSV saved to: {OUTPUT_CSV_PATH}")
    print("Download the CSV file to play it on your Unitree G1 robot or simulator!")
else:
    print("\n❌ Retargeting failed. Check the logs above.")